# PAYROLL_INGEST — Snowflake-native ingestion

Deploys the SharePoint → Snowflake payroll pipeline as a single stored procedure
inside the payroll database. No Dagster dependency, no external orchestrator: the
credential lives in a Snowflake `SECRET`, egress goes through an external access
integration, and every access is visible in `ACCESS_HISTORY`.

## 1. Session context

Point the notebook at the payroll database and schema. Everything below is created
relative to this context, so the same notebook serves dev and production unchanged.

In [ ]:
%%sql -r dataframe_1
USE SCHEMA HR_PAYROLL_DEV.RAW;
USE WAREHOUSE <QIMA_WH>;


## 2. OAuth2 Security Integration 

In [ ]:
%%sql -r dataframe_2
# Create a Network rule
CREATE OR REPLACE NETWORK RULE payroll_graph_network_rule
  MODE = EGRESS
  TYPE = HOST_PORT
  VALUE_LIST = ('login.microsoftonline.com:443', 'graph.microsoft.com:443');

CREATE OR REPLACE SECURITY INTEGRATION payroll_graph_oauth
  TYPE = API_AUTHENTICATION
  AUTH_TYPE = OAUTH2
  ENABLED = TRUE
  OAUTH_CLIENT_ID = '<GRAPH_CLIENT_ID>'
  OAUTH_CLIENT_SECRET = '<GRAPH_CLIENT_SECRET>'
  OAUTH_TOKEN_ENDPOINT = 'https://login.microsoftonline.com/<TENANT_ID>/oauth2/v2.0/token'
  OAUTH_ALLOWED_SCOPES = ('https://graph.microsoft.com/.default');


CREATE OR REPLACE SECRET HR_PAYROLL_DEV.RAW.PAYROLL_GRAPH_SECRET
  TYPE = OAUTH2
  API_AUTHENTICATION = payroll_graph_oauth
  OAUTH_SCOPES = ('https://graph.microsoft.com/.default')
  COMMENT = 'OAuth2 token for SharePoint payroll reader (managed by Snowflake)';

# External access integration: ties network rule + secret together
CREATE OR REPLACE EXTERNAL ACCESS INTEGRATION payroll_graph_eai
  ALLOWED_NETWORK_RULES = (payroll_graph_network_rule)
  ALLOWED_AUTHENTICATION_SECRETS = (HR_PAYROLL_DEV.RAW.PAYROLL_GRAPH_SECRET)
  ENABLED = TRUE;


## 3. Configuration table

In [ ]:
%%sql -r dataframe_3
CREATE TABLE IF NOT EXISTS PIPELINE_CONFIG (
    KEY   VARCHAR NOT NULL PRIMARY KEY,
    VALUE VARCHAR NOT NULL
);

MERGE INTO PIPELINE_CONFIG AS tgt
USING (SELECT * FROM VALUES
    ('SITE_ID',      '<SP_SITE_ID>'),
    ('DRIVE_ID',     '<SP_DRIVE_ID>'),
    ('FOLDER_PATH',  'QIMA_APA_HR-Payroll-Data-Platform-2026'),
    ('SHEET_NAME',   '2026')
) AS src(KEY, VALUE)
ON tgt.KEY = src.KEY
WHEN MATCHED THEN UPDATE SET VALUE = src.VALUE
WHEN NOT MATCHED THEN INSERT (KEY, VALUE) VALUES (src.KEY, src.VALUE);



## 4.Stage and staging table


In [ ]:
%%sql -r dataframe_4
CREATE STAGE IF NOT EXISTS PAYROLL_STAGE
  DIRECTORY = (ENABLE = TRUE)
  COMMENT = 'Raw payroll .xlsx workbooks from SharePoint';

CREATE TABLE IF NOT EXISTS PAYROLL_STAGING (
    LOAD_DATE                                TIMESTAMP_NTZ  NOT NULL,
    SOURCE_FILE_NAME                         VARCHAR        NOT NULL,
    SUBSIDIARY_CODE                          VARCHAR        NOT NULL,
    SOURCE_MODIFIED_DATE                     TIMESTAMP_NTZ  NOT NULL,
    EMPLOYEE_SAP_ID                          VARCHAR,
    SAP_STAFF_NAME                           VARCHAR,
    JOIN_DATE                                VARCHAR,
    LEAVE_DATE                               VARCHAR,
    SUBSIDIARY                               VARCHAR,
    MONTHLY_GROSS_SALARY                     VARCHAR,
    MONTHLY_ALLOWANCE                        VARCHAR,
    EMPLOYER_SOCIAL_CHARGES                  VARCHAR,
    TOTAL_MONTHLY_SALARY_COST                VARCHAR,
    CURRENCY_CONTRACTUAL                     VARCHAR,
    ACTUAL_SALARY_JAN                        VARCHAR,
    ACTUAL_SALARY_FEB                        VARCHAR,
    ACTUAL_SALARY_MAR                        VARCHAR,
    ACTUAL_SALARY_APR                        VARCHAR,
    ACTUAL_SALARY_MAY                        VARCHAR,
    ACTUAL_SALARY_JUN                        VARCHAR,
    ACTUAL_SALARY_JUL                        VARCHAR,
    ACTUAL_SALARY_AUG                        VARCHAR,
    ACTUAL_SALARY_SEP                        VARCHAR,
    ACTUAL_SALARY_OCT                        VARCHAR,
    ACTUAL_SALARY_NOV                        VARCHAR,
    ACTUAL_SALARY_DEC                        VARCHAR,
    TOTAL_SALARY_2026                        VARCHAR,
    ELIGIBLE_YEAR_END_BONUS                  VARCHAR,
    YEAR_END_BONUS_CURRENCY                  VARCHAR,
    MAX_YEAR_END_BONUS                       VARCHAR,
    ACTUAL_YEAR_END_BONUS                    VARCHAR,
    ELIGIBLE_HALF_YEAR_BONUS                 VARCHAR,
    HALF_YEAR_BONUS_CURRENCY                 VARCHAR,
    MAX_HALF_YEAR_BONUS                      VARCHAR,
    ACTUAL_HALF_YEAR_BONUS                   VARCHAR,
    ELIGIBLE_13TH_MONTH                      VARCHAR,
    THIRTEENTH_MONTH_CURRENCY                VARCHAR,
    MAX_13TH_MONTH                           VARCHAR,
    ACTUAL_13TH_MONTH                        VARCHAR,
    ELIGIBLE_HOLIDAY_BONUS                   VARCHAR,
    HOLIDAY_BONUS_CURRENCY                   VARCHAR,
    MAX_HOLIDAY_BONUS                        VARCHAR,
    ACTUAL_HOLIDAY_BONUS                     VARCHAR,
    ELIGIBLE_PROFIT_SHARING                  VARCHAR,
    PROFIT_SHARING_CURRENCY                  VARCHAR,
    MAX_PROFIT_SHARING                       VARCHAR,
    ACTUAL_PROFIT_SHARING                    VARCHAR,
    ELIGIBLE_CCLAB_BONUS                     VARCHAR,
    CCLAB_BONUS_CURRENCY                     VARCHAR,
    MAX_CCLAB_BONUS                          VARCHAR,
    ACTUAL_CCLAB_BONUS                       VARCHAR,
    ELIGIBLE_GRATUITY                        VARCHAR,
    GRATUITY_CURRENCY                        VARCHAR,
    MAX_GRATUITY                             VARCHAR,
    ACTUAL_GRATUITY                          VARCHAR,
    AUDITOR_BONUS_CURRENCY                   VARCHAR,
    AUDITOR_BONUS_Q1                         VARCHAR,
    AUDITOR_BONUS_Q2                         VARCHAR,
    AUDITOR_BONUS_Q3                         VARCHAR,
    AUDITOR_BONUS_Q4                         VARCHAR,
    COMMISSION_CURRENCY                      VARCHAR,
    COMMISSION_Q1                            VARCHAR,
    COMMISSION_Q2                            VARCHAR,
    COMMISSION_Q3                            VARCHAR,
    COMMISSION_Q4                            VARCHAR,
    PAYMENT_TYPE                             VARCHAR,
    CURRENCY_ADHOC                           VARCHAR,
    BONUS_AMOUNT                             VARCHAR,
    AGENCY_NAME                              VARCHAR,
    AGENCY_FEE                               VARCHAR,
    REMARKS                                  VARCHAR
);


## 5. The Ingestion Procedure



In [ ]:
CREATE OR REPLACE PROCEDURE PAYROLL_INGEST()
RETURNS VARIANT
LANGUAGE PYTHON
RUNTIME_VERSION = '3.11'
PACKAGES = ('snowflake-snowpark-python', 'openpyxl', 'requests')
HANDLER = 'main'
EXTERNAL_ACCESS_INTEGRATIONS = (payroll_graph_eai)
SECRETS = ('graph_secret' = HR_PAYROLL_DEV.RAW.PAYROLL_GRAPH_SECRET)
EXECUTE AS OWNER
AS
$$
import re
from datetime import datetime, timezone
from io import BytesIO

import _snowflake
import openpyxl
import requests
from snowflake.snowpark.types import (
    StringType, StructField, StructType, TimestampType,
)


# Frozen header contract (pre-normalised). Any template change fails loudly.
EXPECTED_HEADERS = (
    'Employee SAP ID',
    'SAP Staff Name',
    'Join Date',
    'Leave Date',
    'Subsidiary',
    'Monthly Gross Salary',
    'Monthly Allowance / 2nd part of salary',
    'QIMA (Employer) Social Charges',
    'Total Monthly Salary cost (F+G+H)',
    'Currency',
    'Actual salary Jan 2026',
    'Actual salary Feb 2026',
    'Actual salary Mar 2026',
    'Actual salary Apr 2026',
    'Actual salary May 2026',
    'Actual salary Jun 2026',
    'Actual salary Jul 2026',
    'Actual salary Aug 2026',
    'Actual salary Sept 2026',
    'Actual salary Oct 2026',
    'Actual salary Nov 2026',
    'Actual salary Dec 2026',
    'Total salary 2026',
    'Eligible for Year-end bonus',
    'Year-end bonus (Currency)',
    'Max Year-end bonus amount in 2026',
    'Actual Year-end Bonus amount Paid in 2026',
    'Eligible for Half-year bonus',
    'Half-Year bonus (Currency)',
    'Max Half-year bonus amount in 2026',
    'Actual Half-year Bonus amount Paid in 2026',
    'Eligible for 13th Month salary / Christmas Bonus / Aguinaldo',
    '13th Month salary (Currency)',
    'Max 13th Month salary / Aguinaldo / Christmas Bonus in 2026',
    'Actual 13th Month salary paid in 2026',
    'Eligible for Holiday Bonus',
    'Holiday Bonus (Currency)',
    'Max Holiday Bonus amount in 2026',
    'Actual Holiday Bonus amount paid in 2026',
    'Eligible for Profit Sharing',
    'Profit Sharing (Currency)',
    'Max Profit Sharing amount in 2026',
    'Actual Profit Sharing amount Paid in 2026',
    'Eligible for CCLAB Bonus',
    'CCLAB bonus (Currency)',
    'MaxCCLAB Bonus amount paid in 2026',
    'Actual Bonus amount paid in 2026',
    'Eligible for Gratuity',
    'Gratuity (Currency)',
    'Max Gratuity in 2026',
    'Actual Gratuity paid in 2026',
    'Auditor Bonus (Currency)',
    'Auditor Bonus paid 2026 Q1',
    'Auditor Bonus paid 2026 Q2',
    'Auditor Bonus paid 2026 Q3',
    'Auditor Bonus paid 2026 Q4',
    'Commission (Currency)',
    'Commissions paid 2026 Q1',
    'Commissions paid 2026 Q2',
    'Commissions paid 2026 Q3',
    'Commissions paid 2026 Q4',
    'Payment type',
    'Currency',
    'Bonus amount',
    'Agency Name',
    'Agency Fee',
    'Remarks',
)

COLUMN_COUNT = 67


# ══════════════════════════════════════════════════════════════
# SETUP: config + schema discovery
# ══════════════════════════════════════════════════════════════

def load_config(session):
    rows = session.sql("SELECT KEY, VALUE FROM PIPELINE_CONFIG").collect()
    return {r["KEY"]: r["VALUE"] for r in rows}


def load_columns(session):
    rows = session.sql(
        "SELECT COLUMN_NAME FROM HR_PAYROLL_DEV.INFORMATION_SCHEMA.COLUMNS "
        "WHERE TABLE_SCHEMA = 'RAW' AND TABLE_NAME = 'PAYROLL_STAGING' "
        "ORDER BY ORDINAL_POSITION"
    ).collect()
    all_cols = [r["COLUMN_NAME"] for r in rows]
    return all_cols[:4], all_cols[4:]


# ══════════════════════════════════════════════════════════════
# GRAPH API: list files + download
# ══════════════════════════════════════════════════════════════

def list_files(token, site_id, drive_id, folder_path):
    url = (
        "https://graph.microsoft.com/v1.0/sites/{}/drives/{}/root:/{}:/children"
    ).format(site_id, drive_id, folder_path)
    resp = requests.get(url, headers={"Authorization": "Bearer " + token})
    if resp.status_code != 200:
        raise RuntimeError(
            "list_files failed ({}): {}".format(resp.status_code, resp.text)
        )
    files = [i for i in resp.json().get("value", []) if i["name"].endswith(".xlsx")]
    if not files:
        raise RuntimeError(
            "No .xlsx files in folder. Check FOLDER_PATH in PIPELINE_CONFIG."
        )
    return files


def download(token, drive_id, item_id):
    url = "https://graph.microsoft.com/v1.0/drives/{}/items/{}/content".format(
        drive_id, item_id
    )
    resp = requests.get(url, headers={"Authorization": "Bearer " + token})
    if resp.status_code != 200:
        raise RuntimeError(
            "download failed ({}): {}".format(resp.status_code, resp.text)
        )
    return resp.content


# ══════════════════════════════════════════════════════════════
# PARSER: pure logic — bytes in, rows out
# ══════════════════════════════════════════════════════════════

def _normalise(text):
    if text is None:
        return ""
    return re.sub(r"\s+", " ", str(text)).strip()


def _assert_headers(row2_values):
    actual = [_normalise(v) for v in row2_values[:COLUMN_COUNT]]
    if len(actual) < COLUMN_COUNT:
        raise ValueError(
            "Expected {} columns, got {}.".format(COLUMN_COUNT, len(actual))
        )
    mismatches = [
        "  pos {}: got {!r}, expected {!r}".format(i + 1, a, e)
        for i, (a, e) in enumerate(zip(actual, EXPECTED_HEADERS))
        if a != e
    ]
    if mismatches:
        raise ValueError(
            "Header mismatch at {} column(s):\n{}".format(
                len(mismatches), "\n".join(mismatches[:10])
            )
        )


def subsidiary_code(row):
    value = row[4]
    if not value:
        raise ValueError("Column E (Subsidiary) is empty on a data row.")
    return str(value).split(" - ", 1)[0].strip()


def parse(file_bytes, sheet_name):
    wb = openpyxl.load_workbook(BytesIO(file_bytes), read_only=True, data_only=True)
    try:
        if sheet_name not in wb.sheetnames:
            raise ValueError(
                "Sheet '{}' not found. Available: {}".format(sheet_name, wb.sheetnames)
            )
        ws = wb[sheet_name]
        header_row = next(ws.iter_rows(min_row=2, max_row=2, values_only=True), None)
        if header_row is None:
            raise ValueError("Row 2 (headers) is empty or missing.")
        _assert_headers(header_row)

        rows = []
        for row in ws.iter_rows(min_row=3, values_only=True):
            if row[0] is None:
                continue
            rows.append([
                str(row[i]) if i < len(row) and row[i] is not None else None
                for i in range(COLUMN_COUNT)
            ])
        return rows
    finally:
        wb.close()


# ══════════════════════════════════════════════════════════════
# STAGING: stage file + bulk insert
# ══════════════════════════════════════════════════════════════

def stage_file(session, file_bytes, file_name):
    session.file.put_stream(
        BytesIO(file_bytes),
        "@PAYROLL_STAGE/{}".format(file_name),
        auto_compress=False,
        overwrite=True,
    )


def loaded_versions(session):
    rows = session.sql(
        "SELECT SOURCE_FILE_NAME, MAX(SOURCE_MODIFIED_DATE) "
        "FROM PAYROLL_STAGING GROUP BY 1"
    ).collect()
    return {r[0]: r[1] for r in rows}


def insert_rows(session, rows, file_name, source_modified, meta_cols, source_cols):
    if not rows:
        return 0

    schema = StructType(
        [StructField(meta_cols[0], TimestampType()),
         StructField(meta_cols[1], StringType()),
         StructField(meta_cols[2], StringType()),
         StructField(meta_cols[3], TimestampType())]
        + [StructField(c, StringType()) for c in source_cols]
    )

    load_date = datetime.now(timezone.utc).replace(tzinfo=None)
    data = [
        [load_date, file_name, subsidiary_code(r), source_modified] + r
        for r in rows
    ]

    session.create_dataframe(data, schema=schema).write.save_as_table(
        "PAYROLL_STAGING", mode="append"
    )
    return len(rows)


# ══════════════════════════════════════════════════════════════
# MAIN: entry point — called by CALL PAYROLL_INGEST()
# ══════════════════════════════════════════════════════════════

def main(session):
    # Config from table (no hard-coded values in procedure)
    cfg = load_config(session)

    # Column names from table schema (no hard-coded 67-column tuple)
    meta_cols, source_cols = load_columns(session)

    # OAuth token — Snowflake manages the full client-credentials flow
    token = _snowflake.get_oauth_access_token("graph_secret")

    # List files from SharePoint
    files = list_files(token, cfg["SITE_ID"], cfg["DRIVE_ID"], cfg["FOLDER_PATH"])
    already_loaded = loaded_versions(session)

    success, failed, skipped = [], [], []
    total_rows = 0

    for item in files:
        file_name = item["name"]
        try:
            modified = datetime.fromisoformat(
                item["lastModifiedDateTime"].replace("Z", "+00:00")
            ).replace(tzinfo=None)

            # Skip unchanged files (saves download + parse cost)
            if already_loaded.get(file_name) == modified:
                skipped.append({"file": file_name, "modified": str(modified)})
                continue

            # Download → Stage → Parse → Insert
            file_bytes = download(token, cfg["DRIVE_ID"], item["id"])
            stage_file(session, file_bytes, file_name)
            rows = parse(file_bytes, cfg["SHEET_NAME"])
            count = insert_rows(
                session, rows, file_name, modified, meta_cols, source_cols
            )

            success.append({"file": file_name, "rows": count})
            total_rows += count

        except Exception as e:
            # One bad workbook must not cost the other 31.
            failed.append({"file": file_name, "error": str(e)})

    # Refresh stage directory table (PUT does not update it automatically)
    session.sql("ALTER STAGE PAYROLL_STAGE REFRESH").collect()

    return {
        "success": success,
        "failed": failed,
        "skipped": skipped,
        "total_rows": total_rows,
    }
$$;

## 6. Run the Procedure

In [ ]:
CALL PAYROLL_INGEST();